# Total KM Prediction Model

Predicts estimated range (km) based on battery health, maintenance, age, and other factors.

- CSV file: final_dataset_for_electiccar_2.csv
- Uses a scikit-learn model.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import panel as pn

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

pn.extension(sizing_mode="stretch_width")

DATA_PATH = Path("final_dataset_for_electiccar_2.csv")

df_raw = pd.read_csv(DATA_PATH)

rename_map = {
    "driver id": "driver_id",
    "car id": "car_id",
    "drive name": "drive_name",
    "city": "city",
    "vechicle type": "vehicle_type",
    "vechicle age": "vehicle_age",
    "battery capacity(kw)": "battery_capacity_kw",
    "total km driven": "total_km_driven",
    "current charge percentage": "current_charge_percentage",
    "estimated rangekm": "estimated_rangekm",
    "batery health": "battery_health",
    "vechicle-status": "vehicle_status",
    "average energy per km kwh": "average_energy_per_km_kwh",
    "charging cost": "charging_cost",
    "total maintanace cost": "total_maintenance_cost",
    "max speed": "max_speed",
    "gross revenue": "gross_revenue",
    "driver earnings": "driver_earnings"
}

df = df_raw.rename(columns=rename_map)
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
)

df.head()

,driver_id,car_id,drive_name,city,vehicle_type,vehicle_age,battery_capacity_kw,total_km_driven,current_charge_percentage,estimated_rangekm,battery_health,vehicle_status,average_energy_per_km_kwh,charging_cost,total_maintenance_cost,max_speed,gross_revenue,driver_earnings
0,1,100001,Driver_1,Chennai,Sedan,6.1,40,59686.6,76.3,175.4,96.8,Active,0.174,3157.22,6227.34,120,429016.47,175021.90
1,2,100002,Driver_2,Bangalore,Mini EV,2.2,75,73156.7,88.2,365.5,74.8,Maintenance,0.181,2031.13,32756.08,140,126657.71,65099.30
2,3,100003,Driver_3,Hyderabad,Hatchback,5.0,30,183784.3,58.3,88.3,81.4,Inactive,0.198,1819.30,39736.81,140,483014.11,295301.09
3,4,100004,Driver_4,Chennai,SUV,6.3,30,214552.8,88.0,176.0,83.6,Maintenance,0.150,1231.94,28515.04,100,506848.25,228761.70
4,5,100005,Driver_5,Bangalore,SUV,4.0,50,247433.2,67.6,204.8,90.5,Active,0.165,1530.72,12442.59,140,99641.45,67152.44


In [2]:
target = "estimated_rangekm"

feature_cols = [
    "vehicle_type",
    "city",
    "vehicle_status",
    "vehicle_age",
    "battery_capacity_kw",
    "current_charge_percentage",
    "battery_health",
    "average_energy_per_km_kwh",
    "total_maintenance_cost",
    "max_speed"
]

df_model = df[feature_cols + [target]].dropna()

X = df_model[feature_cols]
y = df_model[target]

categorical_cols = ["vehicle_type", "city", "vehicle_status"]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]
        ), categorical_cols),
        ("num", Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median"))
            ]
        ), numeric_cols),
    ]
)

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", model)
    ]
)

pipeline.fit(X_train, y_train)

preds = pipeline.predict(X_test)
mae = mean_absolute_error(y_test, preds)
rmse = mean_squared_error(y_test, preds, squared=False)
r2 = r2_score(y_test, preds)

metrics = pd.DataFrame(
    {
        "MAE": [mae],
        "RMSE": [rmse],
        "R2": [r2]
    }
)
metrics

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


,MAE,RMSE,R2
0,1.475015,2.183719,0.999603


In [3]:
def series_stats(series):
    return {
        "min": float(series.min()),
        "max": float(series.max()),
        "median": float(series.median())
    }

defaults = {c: series_stats(df_model[c]) for c in numeric_cols}

vehicle_type_widget = pn.widgets.Select(
    name="Vehicle type",
    options=sorted(df_model["vehicle_type"].dropna().unique().tolist())
)
city_widget = pn.widgets.Select(
    name="City",
    options=sorted(df_model["city"].dropna().unique().tolist())
)
status_widget = pn.widgets.Select(
    name="Vehicle status",
    options=sorted(df_model["vehicle_status"].dropna().unique().tolist())
)

vehicle_age_widget = pn.widgets.FloatInput(
    name="Vehicle age",
    value=defaults["vehicle_age"]["median"],
    step=0.1
)
battery_capacity_widget = pn.widgets.FloatInput(
    name="Battery capacity (kw)",
    value=defaults["battery_capacity_kw"]["median"],
    step=1.0
)
charge_widget = pn.widgets.FloatInput(
    name="Current charge (%)",
    value=defaults["current_charge_percentage"]["median"],
    step=1.0
)
battery_health_widget = pn.widgets.FloatInput(
    name="Battery health",
    value=defaults["battery_health"]["median"],
    step=0.1
)
avg_energy_widget = pn.widgets.FloatInput(
    name="Average energy per km (kwh)",
    value=defaults["average_energy_per_km_kwh"]["median"],
    step=0.01
)
maintenance_widget = pn.widgets.FloatInput(
    name="Total maintenance cost",
    value=defaults["total_maintenance_cost"]["median"],
    step=10.0
)
max_speed_widget = pn.widgets.FloatInput(
    name="Max speed",
    value=defaults["max_speed"]["median"],
    step=1.0
)

predict_button = pn.widgets.Button(name="Predict km", button_type="primary")
prediction_pane = pn.pane.Markdown("Ready.")

def on_predict(event):
    input_df = pd.DataFrame({
        "vehicle_type": [vehicle_type_widget.value],
        "city": [city_widget.value],
        "vehicle_status": [status_widget.value],
        "vehicle_age": [vehicle_age_widget.value],
        "battery_capacity_kw": [battery_capacity_widget.value],
        "current_charge_percentage": [charge_widget.value],
        "battery_health": [battery_health_widget.value],
        "average_energy_per_km_kwh": [avg_energy_widget.value],
        "total_maintenance_cost": [maintenance_widget.value],
        "max_speed": [max_speed_widget.value]
    })

    prediction = pipeline.predict(input_df)[0]
    prediction_pane.object = f"**Predicted range:** {prediction:.2f} km"

predict_button.on_click(on_predict)

dashboard = pn.Column(
    pn.pane.Markdown("## Range Prediction"),
    pn.Row(vehicle_type_widget, city_widget, status_widget),
    pn.Row(vehicle_age_widget, battery_capacity_widget, charge_widget),
    pn.Row(battery_health_widget, avg_energy_widget, maintenance_widget, max_speed_widget),
    pn.Row(predict_button),
    prediction_pane,
    sizing_mode="stretch_width"
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'b396dcbe-9490-4947-92aa-f6937667ee97': {'version…